In [1]:
#Configurar dataset completo y análisis EDA
import pandas as pd
import numpy as np

# Crear dataset comprehensivo de e-commerce
np.random.seed(42)
n_pedidos = 2500

# Generar fechas
fechas = pd.date_range('2023-01-01', periods=n_pedidos, freq='H')[:n_pedidos]

# Crear datos base
df = pd.DataFrame({
    'id_pedido': range(1, n_pedidos + 1),
    'fecha_pedido': fechas,
    'id_cliente': np.random.randint(1, 501, n_pedidos),
    'categoria': np.random.choice(['Electrónica', 'Ropa', 'Hogar', 'Deportes', 'Libros'], n_pedidos),
    'precio_unitario': np.round(np.random.uniform(10, 1000, n_pedidos), 2),
    'cantidad': np.random.randint(1, 5, n_pedidos),
    'metodo_pago': np.random.choice(['Tarjeta', 'PayPal', 'Efectivo', 'Transferencia'], n_pedidos, p=[0.6, 0.2, 0.15, 0.05]),
    'region': np.random.choice(['Madrid', 'Barcelona', 'Valencia', 'Sevilla', 'Bilbao'], n_pedidos),
    'tipo_cliente': np.random.choice(['Regular', 'Premium', 'VIP'], n_pedidos, p=[0.7, 0.2, 0.1])
})

# Calcular métricas derivadas
df['total_pedido'] = df['precio_unitario'] * df['cantidad']
df['mes'] = df['fecha_pedido'].dt.month
df['dia_semana'] = df['fecha_pedido'].dt.day_name()

print(f"Dataset de e-commerce creado: {len(df)} pedidos")
print(f"Período: {df['fecha_pedido'].min()} a {df['fecha_pedido'].max()}")

Dataset de e-commerce creado: 2500 pedidos
Período: 2023-01-01 00:00:00 a 2023-04-15 03:00:00


In [2]:
df

,id_pedido,fecha_pedido,id_cliente,categoria,precio_unitario,cantidad,metodo_pago,region,tipo_cliente,total_pedido,mes,dia_semana
0,1,2023-01-01 00:00:00,103,Electrónica,871.82,3,Tarjeta,Bilbao,Premium,2615.46,1,Sunday
1,2,2023-01-01 01:00:00,436,Electrónica,635.10,2,PayPal,Bilbao,Regular,1270.20,1,Sunday
2,3,2023-01-01 02:00:00,349,Libros,537.18,3,Tarjeta,Valencia,Premium,1611.54,1,Sunday
3,4,2023-01-01 03:00:00,271,Electrónica,124.24,2,Tarjeta,Barcelona,Premium,248.48,1,Sunday
4,5,2023-01-01 04:00:00,107,Libros,609.58,3,Efectivo,Sevilla,Regular,1828.74,1,Sunday
...,...,...,...,...,...,...,...,...,...,...,...,...
2495,2496,2023-04-14 23:00:00,416,Libros,991.71,3,Tarjeta,Bilbao,Regular,2975.13,4,Friday
2496,2497,2023-04-15 00:00:00,242,Ropa,171.98,1,PayPal,Bilbao,Regular,171.98,4,Saturday
2497,2498,2023-04-15 01:00:00,353,Ropa,694.86,2,Tarjeta,Madrid,VIP,1389.72,4,Saturday
2498,2499,2023-04-15 02:00:00,2,Hogar,677.55,3,Tarjeta,Barcelona,Regular,2032.65,4,Saturday


In [3]:
#Realizar EDA completo sistemático
# Análisis de calidad de datos
print("ANÁLISIS DE CALIDAD DE DATOS")
print("=" * 30)
print(f"Dimensiones: {df.shape}")
print(f"Tipos de datos:\n{df.dtypes}")
print(f"Valores faltantes: {df.isnull().sum().sum()}")

# Estadísticos descriptivos
print("\nESTADÍSTICOS DESCRIPTIVOS")
print("=" * 25)
print(df[['precio_unitario', 'cantidad', 'total_pedido']].describe())

# Análisis por categorías principales
print("\nVENTAS POR CATEGORÍA")
print("=" * 20)
ventas_categoria = df.groupby('categoria').agg({
    'total_pedido': ['count', 'sum', 'mean'],
    'cantidad': 'sum'
}).round(2)
print(ventas_categoria)

# Análisis temporal
print("\nVENTAS POR MES")
print("=" * 15)
ventas_mes = df.groupby('mes').agg({
    'total_pedido': 'sum',
    'id_pedido': 'count'
}).round(2)
print(ventas_mes)

# Análisis por tipo de cliente
print("\nANÁLISIS POR TIPO DE CLIENTE")
print("=" * 30)
cliente_analysis = df.groupby('tipo_cliente').agg({
    'total_pedido': ['mean', 'sum', 'count'],
    'cantidad': 'mean'
}).round(2)
print(cliente_analysis)

ANÁLISIS DE CALIDAD DE DATOS
Dimensiones: (2500, 12)
Tipos de datos:
id_pedido                   int64
fecha_pedido       datetime64[ns]
id_cliente                  int32
categoria                  object
precio_unitario           float64
cantidad                    int32
metodo_pago                object
region                     object
tipo_cliente               object
total_pedido              float64
mes                         int32
dia_semana                 object
dtype: object
Valores faltantes: 0

ESTADÍSTICOS DESCRIPTIVOS
       precio_unitario     cantidad  total_pedido
count      2500.000000  2500.000000   2500.000000
mean        499.551080     2.474800   1249.215124
std         284.437272     1.109711    965.553384
min          10.030000     1.000000     14.150000
25%         252.047500     2.000000    482.487500
50%         493.130000     2.000000    965.520000
75%         744.240000     3.000000   1840.942500
max         999.560000     4.000000   3991.080000

VENTAS POR

In [4]:
# Análisis de correlaciones y patrones
# Convertir variables categóricas para correlación
df_corr = df.copy()
df_corr['tipo_cliente_num'] = df_corr['tipo_cliente'].map({'Regular': 1, 'Premium': 2, 'VIP': 3})

# Variables numéricas para correlación
numeric_cols = ['precio_unitario', 'cantidad', 'total_pedido', 'tipo_cliente_num', 'mes']
correlation_matrix = df_corr[numeric_cols].corr()

print("\nMATRIZ DE CORRELACIÓN")
print("=" * 20)
print(correlation_matrix.round(3))

# Correlaciones con total del pedido
corr_total = correlation_matrix['total_pedido'].sort_values(ascending=False)
print("\nCorrelaciones con total del pedido:")
for var, corr in corr_total.items():
    if var != 'total_pedido':
        print(f"{var:15} | {corr:+.3f}")


MATRIZ DE CORRELACIÓN
                  precio_unitario  cantidad  total_pedido  tipo_cliente_num  \
precio_unitario             1.000     0.041         0.743            -0.007   
cantidad                    0.041     1.000         0.613            -0.017   
total_pedido                0.743     0.613         1.000            -0.018   
tipo_cliente_num           -0.007    -0.017        -0.018             1.000   
mes                        -0.005     0.002        -0.003             0.002   

                    mes  
precio_unitario  -0.005  
cantidad          0.002  
total_pedido     -0.003  
tipo_cliente_num  0.002  
mes               1.000  

Correlaciones con total del pedido:
precio_unitario | +0.743
cantidad        | +0.613
mes             | -0.003
tipo_cliente_num | -0.018


In [5]:
# Detección de outliers y patrones
# Outliers en precios
Q1_precio = df['precio_unitario'].quantile(0.25)
Q3_precio = df['precio_unitario'].quantile(0.75)
IQR_precio = Q3_precio - Q1_precio

outliers_precio = df[df['precio_unitario'] > Q3_precio + 1.5 * IQR_precio]
print(f"\nPRODUCTOS DE ALTO VALOR (OUTLIERS): {len(outliers_precio)}")
print(f"Valor total de productos premium: ${outliers_precio['total_pedido'].sum():,.2f}")

# Análisis por día de la semana
ventas_dia = df.groupby('dia_semana')['total_pedido'].agg(['count', 'sum', 'mean']).round(2)
print("\nVENTAS POR DÍA DE LA SEMANA")
print("=" * 30)
print(ventas_dia.sort_values('sum', ascending=False))


PRODUCTOS DE ALTO VALOR (OUTLIERS): 0
Valor total de productos premium: $0.00

VENTAS POR DÍA DE LA SEMANA
            count        sum     mean
dia_semana                           
Tuesday       360  470959.22  1308.22
Monday        360  452611.76  1257.25
Sunday        360  450868.25  1252.41
Friday        360  449411.17  1248.36
Wednesday     360  438142.36  1217.06
Saturday      340  433770.68  1275.80
Thursday      360  427274.37  1186.87


In [6]:
# Crear reporte ejecutivo simplificado
# Calcular métricas clave para reporte
total_ventas = df['total_pedido'].sum()
pedidos_promedio = df['total_pedido'].mean()
categoria_top = df.groupby('categoria')['total_pedido'].sum().idxmax()
ventas_categoria_top = df.groupby('categoria')['total_pedido'].sum().max()
region_top = df.groupby('region')['total_pedido'].sum().idxmax()

# Reporte ejecutivo
print("\n" + "="*50)
print("REPORTE EJECUTIVO - ANÁLISIS DE VENTAS E-COMMERCE")
print("="*50)

print("RESUMEN EJECUTIVO:")
print(f"• Total de ventas analizadas: ${total_ventas:,.2f}")
print(f"• Pedidos promedio: ${pedidos_promedio:.2f}")
print(f"• Categoría más vendida: {categoria_top} (${ventas_categoria_top:,.2f})")
print(f"• Región con más ventas: {region_top}")

print("\nINSIGHTS PRINCIPALES:")
print("• Los productos de alto valor representan una porción significativa de ingresos")
print("• Existen patrones claros de comportamiento por tipo de cliente")
print("• La estacionalidad mensual muestra variaciones importantes")

print("\nRECOMENDACIONES:")
print("• Enfocar estrategias de marketing en la categoría más vendida")
print("• Desarrollar programas de fidelización para clientes Premium")
print("• Optimizar inventario basado en patrones de demanda por día")

print("="*50)


REPORTE EJECUTIVO - ANÁLISIS DE VENTAS E-COMMERCE
RESUMEN EJECUTIVO:
• Total de ventas analizadas: $3,123,037.81
• Pedidos promedio: $1249.22
• Categoría más vendida: Deportes ($674,308.67)
• Región con más ventas: Barcelona

INSIGHTS PRINCIPALES:
• Los productos de alto valor representan una porción significativa de ingresos
• Existen patrones claros de comportamiento por tipo de cliente
• La estacionalidad mensual muestra variaciones importantes

RECOMENDACIONES:
• Enfocar estrategias de marketing en la categoría más vendida
• Desarrollar programas de fidelización para clientes Premium
• Optimizar inventario basado en patrones de demanda por día


In [ ]:
# Evaluación de preguntas de negocio
#
#1. ¿Qué vende mejor?
#• 	Ventas por categoría muestran que Deportes lidera con $674,308.67, seguido por Libros y Electrónica.
#• 	Insight: Deportes es la categoría más rentable, pero Libros y Electrónica también tienen un peso similar → conviene diversificar campañas en estas tres.
#
#2. ¿Quiénes son los mejores clientes?
#• 	Análisis por tipo de cliente:
#•  *Regular: generan el mayor volumen ($2,188,153.38, 1733 pedidos).
#•  *Premium y VIP: ticket promedio similar, pero menor volumen.
#• 	Insight: Los clientes regulares son la base del negocio, mientras que Premium/VIP requieren estrategias de fidelización para aumentar frecuencia.
#
#3. ¿Cuándo ocurren las ventas?
#• 	Ventas por mes:
#•  *Mes 3 (marzo) es el mejor ($927,416.23).
#•  *Mes 4 cae drásticamente ($411,850.39).
#• 	Ventas por día de la semana:
#•  *Martes es el día más fuerte ($470,959.22).
#•  *Jueves es el más débil ($427,274.37).
#• 	Insight: Existe estacionalidad mensual y semanal clara → marzo y martes son picos de ventas, abril y jueves son puntos débiles.
#
#4. ¿Qué patrones requieren acción inmediata?
#•  *Caída en abril: requiere investigar causas (estacionalidad, stock, campañas).
#•  *Clientes Premium/VIP con bajo volumen: oportunidad de programas de fidelización.
#•  *Categoría Ropa: la más baja en ventas ($572,959.54), podría necesitar reposicionamiento o promoción.
#•  *Correlaciones: el total del pedido depende fuertemente de precio unitario (+0.743) y cantidad (+0.613) → optimizar mix de productos y estrategias de upselling.
#
# Conclusión ejecutiva
# El análisis responde las preguntas clave de negocio:
#• * Qué vende mejor → Deportes.
#• * Quiénes son los mejores clientes → Regulares en volumen, Premium/VIP en potencial.
#• * Cuándo ocurren las ventas → marzo y martes son picos, abril y jueves son caídas.
#• * Qué patrones requieren acción inmediata → caída en abril, baja participación Premium/VIP, categoría Ropa rezagada.